<div style="background-color: #0b1f1a; padding: 25px; border-left: 10px solid #00f59f; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.2);">
    <h1 style="color: #ffffff; margin: 0; font-family: 'Segoe UI', Arial, sans-serif; font-size: 32px;">
        📈 Marketing Performance Page Validation
    </h1>
    <p style="color: #b0c9c2; margin: 10px 0 0 0; font-size: 16px; font-family: 'Segoe UI', Arial, sans-serif;">
        Automated SQL Data Extraction & Marketing Performance Verification Pipeline
    </p>
</div>

In [1]:
import os
import boto3
import duckdb
import pandas as pd
import dataframe_image as dfi
from io import BytesIO
from dotenv import load_dotenv

# 1. Load AWS Credentials from .env
load_dotenv(dotenv_path="../../.env")
AWS_REGION = os.getenv("AWS_REGION")
PROCESSED_BUCKET = 'ra2-processed-data'

# 2. Setup marketing_performance_output folder
marketing_output_folder = "../../eda_outputs/marketing_performance_output"
os.makedirs(marketing_output_folder, exist_ok=True)

# 3. Connect to S3 securely
s3 = boto3.client('s3', region_name=AWS_REGION)
print("🔗 Connecting to S3 Processed Bucket for Marketing Performance...\n")

# 4. Download ALL files from S3 and explicitly register them in DuckDB
response = s3.list_objects_v2(Bucket=PROCESSED_BUCKET)

print("📥 Downloading and Registering Clean Tables:")
for obj in response.get('Contents', []):
    file_key = obj['Key']
    
    if file_key.endswith('.csv'):
        table_name = file_key.replace('.csv', '').lower()
        print(f"   -> Fetching {file_key} and registering as table: '{table_name}'...")
        
        csv_obj = s3.get_object(Bucket=PROCESSED_BUCKET, Key=file_key)
        df_temp = pd.read_csv(BytesIO(csv_obj['Body'].read()))
        
        # Explicitly register the dataframe into DuckDB
        duckdb.register(table_name, df_temp)

print("\n✅ All clean data successfully registered in DuckDB for Marketing Performance!")

# 5. DuckDB Validation Function (Saves to Marketing Performance Output Folder)
def save_sql_to_image(sql_query, image_name):
    print(f"Executing query for: {image_name}...")
    df = duckdb.query(sql_query).df()
    
    file_path = os.path.join(marketing_output_folder, f"{image_name}.png")
    dfi.export(df, file_path, table_conversion="matplotlib")
    
    print(f"✅ Image saved successfully at: {file_path}\n")
    return df

c:\Users\pushp\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.2.3) or chardet (7.6.0)/charset_normalizer (3.4.0) doesn't match a supported version!
  warnings.warn(


🔗 Connecting to S3 Processed Bucket for Marketing Performance...

📥 Downloading and Registering Clean Tables:
   -> Fetching customers.csv and registering as table: 'customers'...
   -> Fetching delivery.csv and registering as table: 'delivery'...
   -> Fetching feedback.csv and registering as table: 'feedback'...
   -> Fetching menu_items.csv and registering as table: 'menu_items'...
   -> Fetching order_items.csv and registering as table: 'order_items'...
   -> Fetching orders.csv and registering as table: 'orders'...
   -> Fetching outlets.csv and registering as table: 'outlets'...
   -> Fetching promotions.csv and registering as table: 'promotions'...

✅ All clean data successfully registered in DuckDB for Marketing Performance!


In [2]:
import os
import dataframe_image as dfi

# --- 1. Promo Redemption Rate KPI ---
df_promo_rate = duckdb.query("""
    SELECT ROUND(
        (COUNT(CASE WHEN promo_id IS NOT NULL THEN 1 END) * 100.0) / COUNT(*), 2
    ) AS "Promo Redemption Rate (%)"
    FROM orders;
""").df()
dfi.export(df_promo_rate, os.path.join(marketing_output_folder, "1_kpi_promo_redemption_rate.png"), table_conversion="matplotlib")


# --- 2. Repeat Customer Rate KPI ---
df_repeat_rate = duckdb.query("""
    WITH customer_orders AS (
        SELECT customer_id, COUNT(order_id) as total_orders
        FROM orders
        GROUP BY customer_id
    )
    SELECT ROUND(
        (COUNT(CASE WHEN total_orders > 1 THEN 1 END) * 100.0) / COUNT(*), 2
    ) AS "Repeat Customer Rate (%)"
    FROM customer_orders;
""").df()
dfi.export(df_repeat_rate, os.path.join(marketing_output_folder, "2_kpi_repeat_customer_rate.png"), table_conversion="matplotlib")


# --- 3. Total Customers KPI ---
df_total_cust = duckdb.query("""
    SELECT COUNT(customer_id) AS "Total Customers"
    FROM customers;
""").df()
dfi.export(df_total_cust, os.path.join(marketing_output_folder, "3_kpi_total_customers.png"), table_conversion="matplotlib")


# --- 4. Average CLV (Customer Lifetime Value) KPI ---
df_avg_clv = duckdb.query("""
    SELECT ROUND(AVG(total_lifetime_value), 2) AS "Average CLV (₹)"
    FROM customers;
""").df()
dfi.export(df_avg_clv, os.path.join(marketing_output_folder, "4_kpi_average_clv.png"), table_conversion="matplotlib")


# --- 5. Churned Customers KPI ---
# Assuming customers with no orders in a recent window or based on status/activity
df_churned = duckdb.query("""
    SELECT COUNT(customer_id) AS "Churned Customers"
    FROM customers
    WHERE total_lifetime_value = 0 OR customer_id NOT IN (
        SELECT DISTINCT customer_id FROM orders WHERE order_date >= '2026-01-01'
    );
""").df()
dfi.export(df_churned, os.path.join(marketing_output_folder, "5_kpi_churned_customers.png"), table_conversion="matplotlib")

print("✅ All 5 Marketing Performance KPIs successfully generated and saved!")

✅ All 5 Marketing Performance KPIs successfully generated and saved!


In [3]:
import matplotlib.pyplot as plt
import os

# 1. Query monthly new customers and calculate monthly repeat customer rate %
df_main_marketing = duckdb.query("""
    WITH monthly_stats AS (
        SELECT 
            strftime('%b', CAST(o.order_date AS DATE)) AS month_name,
            MONTH(CAST(o.order_date AS DATE)) AS month_num,
            COUNT(DISTINCT c.customer_id) AS new_customers
        FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id
        WHERE o.order_date IS NOT NULL
        GROUP BY month_name, month_num
    ),
    repeat_rates AS (
        SELECT 
            strftime('%b', CAST(order_date AS DATE)) AS month_name,
            MONTH(CAST(order_date AS DATE)) AS month_num,
            ROUND(
                (COUNT(CASE WHEN customer_id IN (
                    SELECT customer_id FROM orders o2 WHERE o2.order_date < orders.order_date
                ) THEN 1 END) * 100.0) / COUNT(*), 2
            ) as repeat_rate
        FROM orders
        WHERE order_date IS NOT NULL
        GROUP BY month_name, month_num
    )
    SELECT 
        m.month_name,
        m.new_customers,
        COALESCE(r.repeat_rate, 30.0) AS repeat_rate
    FROM monthly_stats m
    LEFT JOIN repeat_rates r ON m.month_num = r.month_num
    ORDER BY m.month_num ASC;
""").df()

# 2. Plot Dual-Axis Combo Chart
plt.figure(figsize=(12, 6), facecolor='#0b1f1a')
ax1 = plt.axes()
ax1.set_facecolor('#0b1f1a')

x_indices = range(len(df_main_marketing))

# Left Y-axis: New Customers Bars (Neon Mint)
bars = ax1.bar(x_indices, df_main_marketing['new_customers'], color='#00f59f', width=0.6, alpha=0.85)
ax1.set_ylabel('New Customers', color='#b0c9c2', fontsize=11, fontweight='bold')
ax1.tick_params(axis='y', colors='#b0c9c2')
ax1.set_ylim(0, max(df_main_marketing['new_customers']) * 1.25)

# Right Y-axis: Repeat Customer Rate % Line (White with markers)
ax2 = ax1.twinx()
ax2.set_facecolor('#0b1f1a')
line = ax2.plot(x_indices, df_main_marketing['repeat_rate'], color='white', marker='o', linewidth=2.5, markersize=6)
ax2.set_ylabel('Repeat Customer Rate %', color='#b0c9c2', fontsize=11, fontweight='bold')
ax2.tick_params(axis='y', colors='#b0c9c2')
ax2.set_ylim(25, 40)

# X-Axis Styling
ax1.set_xticks(x_indices)
ax1.set_xticklabels(df_main_marketing['month_name'], color='#b0c9c2', fontsize=10)

# Title and Grid
plt.title('Monthly new and returning customers', color='white', fontsize=15, fontweight='bold', pad=15)
ax1.grid(axis='y', linestyle='--', alpha=0.15, color='#b0c9c2')

# Border Spines Styling
for spine in ax1.spines.values():
    spine.set_color('#2e5e52')
for spine in ax2.spines.values():
    spine.set_color('#2e5e52')

plt.tight_layout()

# 3. Save Image
file_path = os.path.join(marketing_output_folder, "6_main_chart_monthly_new_returning_customers.png")
plt.savefig(file_path, facecolor='#0b1f1a', edgecolor='none', dpi=300)
plt.close()

print(f"✅ Marketing Performance Main Chart successfully generated and saved at: {file_path}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Marketing Performance Main Chart successfully generated and saved at: ../../eda_outputs/marketing_performance_output\6_main_chart_monthly_new_returning_customers.png


In [4]:
import matplotlib.pyplot as plt
import os

# 1. Query monthly new customers and calculate monthly repeat customer rate %
df_main_marketing = duckdb.query("""
    WITH monthly_stats AS (
        SELECT 
            strftime('%b', CAST(o.order_date AS DATE)) AS month_name,
            MONTH(CAST(o.order_date AS DATE)) AS month_num,
            COUNT(DISTINCT c.customer_id) AS new_customers
        FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id
        WHERE o.order_date IS NOT NULL
        GROUP BY month_name, month_num
    ),
    repeat_rates AS (
        SELECT 
            strftime('%b', CAST(order_date AS DATE)) AS month_name,
            MONTH(CAST(order_date AS DATE)) AS month_num,
            ROUND(
                (COUNT(CASE WHEN customer_id IN (
                    SELECT customer_id FROM orders o2 WHERE o2.order_date < orders.order_date
                ) THEN 1 END) * 100.0) / COUNT(*), 2
            ) as repeat_rate
        FROM orders
        WHERE order_date IS NOT NULL
        GROUP BY month_name, month_num
    )
    SELECT 
        m.month_name,
        m.new_customers,
        COALESCE(r.repeat_rate, 30.0) AS repeat_rate
    FROM monthly_stats m
    LEFT JOIN repeat_rates r ON m.month_num = r.month_num
    ORDER BY m.month_num ASC;
""").df()

# 2. Plot Dual-Axis Combo Chart
plt.figure(figsize=(12, 6), facecolor='#0b1f1a')
ax1 = plt.axes()
ax1.set_facecolor('#0b1f1a')

x_indices = range(len(df_main_marketing))

# Left Y-axis: New Customers Bars (Neon Mint)
bars = ax1.bar(x_indices, df_main_marketing['new_customers'], color='#00f59f', width=0.6, alpha=0.85)
ax1.set_ylabel('New Customers', color='#b0c9c2', fontsize=11, fontweight='bold')
ax1.tick_params(axis='y', colors='#b0c9c2')
ax1.set_ylim(0, max(df_main_marketing['new_customers']) * 1.25)

# Right Y-axis: Repeat Customer Rate % Line (White with markers)
ax2 = ax1.twinx()
ax2.set_facecolor('#0b1f1a')
line = ax2.plot(x_indices, df_main_marketing['repeat_rate'], color='white', marker='o', linewidth=2.5, markersize=6)
ax2.set_ylabel('Repeat Customer Rate %', color='#b0c9c2', fontsize=11, fontweight='bold')
ax2.tick_params(axis='y', colors='#b0c9c2')
ax2.set_ylim(25, 40)

# X-Axis Styling
ax1.set_xticks(x_indices)
ax1.set_xticklabels(df_main_marketing['month_name'], color='#b0c9c2', fontsize=10)

# Title and Grid
plt.title('Monthly new and returning customers', color='white', fontsize=15, fontweight='bold', pad=15)
ax1.grid(axis='y', linestyle='--', alpha=0.15, color='#b0c9c2')

# Border Spines Styling
for spine in ax1.spines.values():
    spine.set_color('#2e5e52')
for spine in ax2.spines.values():
    spine.set_color('#2e5e52')

plt.tight_layout()

# 3. Save Image
file_path = os.path.join(marketing_output_folder, "6_main_chart_monthly_new_returning_customers.png")
plt.savefig(file_path, facecolor='#0b1f1a', edgecolor='none', dpi=300)
plt.close()

print(f"✅ Marketing Performance Main Chart successfully generated and saved at: {file_path}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Marketing Performance Main Chart successfully generated and saved at: ../../eda_outputs/marketing_performance_output\6_main_chart_monthly_new_returning_customers.png
